## VINS Dataset: Detecting UI Elements for Accessibility 
VINS dataset is an annotated dataset containing a representative collection of UI screens across two design stages: abstract wireframes and high-fidelity fully designed interfaces. All of these UIs are annotated with bounding boxes spanning different classes of UI components. We identified a total of 11 UI components with varying functionality: background images, sliding menus, pop-up windows, input fields, icons, images, texts, switches, checked views, text buttons, and page indicators. Based on our analysis and due to relatively small training instances, we combined radio buttons and checkboxes to represent the checked view class.

VINS dataset has a total of 4,800 images of UI screens of different design stages to ensure that the VINS can perform on a wider variety of design inputs. It includes the following:

- 257 images of abstract wireframes UIs

- 4,543 images of high-fidelity screens:
    - 2000 images of Android screen collected from Rico Dataset
    - 740 images of new Android screens
    - 1200 images of iphone screens
    - 603 images of UI design collected from design sharing websites


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
torchvision.disable_beta_transforms_warning()
import torchvision.transforms.v2 as transforms
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineRenderer.figure_format = 'retina'

import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import  utils

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

plt.ion()   # interactive mode

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [7]:
# took dataset formatting help from https://www.learnpytorch.io/04_pytorch_custom_datasets/
import zipfile
from pathlib import Path

# Setup path to data folder
data_path = Path("/home/lurena/scratch/")
image_path = data_path / "VINS_dataset"

# If the image folder doesn't exist, download it and prepare it... 
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

    # Unzip VINS file
    with zipfile.ZipFile(data_path / "VINS_dataset.zip", "r") as zip_ref:
        print("Unzipping VINS file...") 
        zip_ref.extractall(image_path)

/home/lurena/scratch/VINS_dataset directory exists.


In [9]:
import os
def walk_through_dir(dir_path):
  """
  Walks through dir_path returning its contents.
  Args:
    dir_path (str or pathlib.Path): target directory
  
  Returns:
    A print out of:
      number of subdiretories in dir_path
      number of images (files) in each subdirectory
      name of each subdirectory
  """
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")


In [11]:
#walk_through_dir(image_path)

In [21]:
transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ConvertImageDtype(),
])

import random

'''
Took inspo from 
https://github.com/mrdbourke/pytorch-deep-learning/blob/main/extras/04_custom_data_creation.ipynb
in splitting my data up, since VINS isn't included in 
pytorch's datasets
'''

# Set training/test/validate sizes
train_size = 0.6
test_size = 1-(train_size+0.1) #0.1 here defines valid_size tbh
valid_size = 1-(train_size+test_size)

if train_size + test_size + valid_size > 1:
    raise ValueError("Invalid data split sizes")

def get_subset(ip=image_path, data_splits=["train","test","valid"],
              amount=[train_size,test_size,valid_size],seed=42): 
    random.seed(42)
    label_splits = {}

    # Get labels
    i=0
    for data_split in data_splits:
        label_dir_name = ip / f"{data_split}.txt"
        if not label_dir_name.isdir():
            print(f"[INFO] Creating image path for: {data_split}...")
            label_path = pathlib.Path(label_dir_name)
            label_path.mkdir(parents=True, exit_ok=True)
            
            with open(ip, "r") as f:
                labels = [line.strip("\n") for line in f.readlines()]

            # Get random subset of image ID's
            num_sample = round(amount[i]*len(labels))
            print(f"[INFO] Getting random subset of {num_sample} images for {data_split}...")
            sampled_images = random.sample(labels, k=num_sample)
    
            # Apply full paths
            im_paths = [pathlib.Path(str(ip / sample_image) + ".jpg") for sample_image in sampled_images]
            label_splits[data_split] = im_paths
            
        else: 
            print(f"[INFO] Path already exists for: {data_split}. Continuing...")
        i += 1
    return label_splits

In [22]:
classes = ['bg_image', 'sld_menu','pop_up','input_fld','icon','image','txt','switch','chck_view','txt_button','page_indicator']
#radio buttons and checkboxes represented under checked view class

#current split: train 0.6, test 0.3, valid 0.1
label_splits = get_subset()

AttributeError: 'PosixPath' object has no attribute 'isdir'

In [ ]:
#actually copying data over to train/test/valid folders.....

import shutil

for image_split in label_splits.keys():
    for im_path in label_splits[str(image_split)]:
        pass
    pass


'''
import shutil

for image_split in label_splits.keys():
    for image_path in label_splits[str(image_split)]:
        dest_dir = target_dir / image_split / image_path.parent.stem / image_path.name
        if not dest_dir.parent.is_dir():
            dest_dir.parent.mkdir(parents=True, exist_ok=True)
        print(f"[INFO] Copying {image_path} to {dest_dir}...")
        shutil.copy2(image_path, dest_dir)

from https://github.com/mrdbourke/pytorch-deep-learning/blob/main/extras/04_custom_data_creation.ipynb
'''